# Oral screen shift: the whole run in Colab

Downloads the datasets, builds the splits, trains every model and saves the predictions. Nothing is downloaded to your own computer except one small zip at the end.

Before running: **Runtime > Change runtime type > T4 GPU**.

If Colab disconnects, run the cells again from the top: finished work is skipped.

## 1. Code, Google Drive, GPU check

Drive stores the splits, features, trained models and predictions (about 2 GB), so a disconnect does not lose them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf oral-screen-shift
!git clone -q https://github.com/sachchi-sapiens/oral-screen-shift.git
%cd /content/oral-screen-shift
!pip install -q timm

# keep everything that takes time to make on Drive
!mkdir -p /content/drive/MyDrive/oral-screen-shift/splits /content/drive/MyDrive/oral-screen-shift/features /content/drive/MyDrive/oral-screen-shift/checkpoints /content/drive/MyDrive/oral-screen-shift/preds results
!cp -n splits/*.csv /content/drive/MyDrive/oral-screen-shift/splits/ 2>/dev/null; rm -rf splits results/preds
!ln -sfn /content/drive/MyDrive/oral-screen-shift/splits splits
!ln -sfn /content/drive/MyDrive/oral-screen-shift/features features
!ln -sfn /content/drive/MyDrive/oral-screen-shift/checkpoints checkpoints
!ln -sfn /content/drive/MyDrive/oral-screen-shift/preds results/preds

import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))

## 2. Check the code works here

In [ ]:
!python -m pytest -q

## 3. Download the four open datasets (about 10 GB, onto Colab's own disk)

In [ ]:
for site in ['smartom', 'china', 'karnataka', 'code']:
    !python scripts/download.py --site {site} --out /content/data

## 4. The Sri Lanka dataset (restricted)

Your Zenodo account has been given access. Zenodo does not hand restricted files to a script reliably, so the sure way is to pass them through your Google Drive:

1. Sign in at zenodo.org and open https://zenodo.org/records/10664056
2. Under **Files**, download all four: `Images.zip`, `Annotation.json`, `Imagewise_data.csv`, `Patientwise_data.csv`.
3. In Google Drive open the folder `oral-screen-shift`, make a folder named `srilanka` inside it, and upload the four files there.
4. Run the cell below. You can then delete the four files from your computer.

If the Drive folder is empty, the cell tries a Zenodo token instead (Colab key icon > secret named `ZENODO_TOKEN`). Never paste a token into a cell or share it.

In [ ]:
import glob, os
src = '/content/drive/MyDrive/oral-screen-shift/srilanka'
if glob.glob(src + '/*'):
    !mkdir -p /content/data/srilanka
    !cp {src}/*.csv {src}/*.json /content/data/srilanka/
    !for z in {src}/*.zip; do unzip -q -o "$$z" -d /content/data/srilanka; done
    print('copied from Google Drive')
else:
    from google.colab import userdata
    os.environ['ZENODO_TOKEN'] = userdata.get('ZENODO_TOKEN')
    !python scripts/download.py --site srilanka --out /content/data

### Look inside (first run only)

Copy the output of this cell and send it back, so the loader for this dataset can be written. It prints file names and column names, and three rows of the image table. It prints no patient details.

In [ ]:
import json, os
import pandas as pd
root = '/content/data/srilanka'
for folder, _, files in os.walk(root):
    print(folder.replace(root, '.') or '.', '|', len(files), 'files |', sorted(files)[:4])
for name in sorted(os.listdir(root)):
    if name.lower().endswith('.csv'):
        table = pd.read_csv(os.path.join(root, name))
        print()
        print(name, table.shape, 'columns:', list(table.columns))
        if 'image' in name.lower():
            print(table.head(3).to_string())
            for column in table.columns:
                if table[column].nunique() <= 12:
                    print(column, table[column].value_counts().to_dict())
    if name.lower().endswith('.json'):
        ann = json.load(open(os.path.join(root, name)))
        print()
        print(name, {k: (len(v) if isinstance(v, list) else v) for k, v in ann.items()})
        print('image entry:', ann['images'][0])
        print('categories:', ann.get('categories'))

---
**Stop here on the first run** and send back the output above. Run the cells below once you are told the Sri Lanka loader is in the repository (then run cell 1 again first, to fetch it).

## 5. Index the photos, find duplicates, write the splits (about 15 minutes, once)

In [ ]:
import os
if not os.path.exists('splits/seed0.restricted.csv'):
    !python scripts/prepare.py --data /content/data
else:
    print('splits already made')

## 6. Make small copies of the photos so training is fast

In [ ]:
!python scripts/shrink.py --data /content/data --out /content/data_small

## 7. Features and linear probes

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage features
!python -m experiments.run_gpu --data /content/data_small --stage probe

## 8. Baseline ResNet-50

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage resnet

## 9. ResNet-50 with colour augmentation

In [ ]:
!python -m experiments.run_gpu --data /content/data_small --stage resnet_color

## 10. Tables and figures, then pack the results

This writes `oral-screen-shift-results.zip` to the `oral-screen-shift` folder in your Google Drive (a few MB). Download it and unzip it into the project folder on your computer.

In [ ]:
!python -m experiments.e1_matrix
!python -m experiments.e2_fixes
!python -m experiments.e3_refusal
!cp results/*.csv results/*.png /content/drive/MyDrive/oral-screen-shift/ 2>/dev/null
!cd /content/drive/MyDrive/oral-screen-shift && rm -f oral-screen-shift-results.zip && zip -q -r oral-screen-shift-results.zip splits preds *.csv *.png && ls -lh oral-screen-shift-results.zip